# The Results
- Features engineered: bed to bath ratio, age
Top 20 feature importances (Random Forest):
LivingArea                                 0.379781
YearBuilt                                  0.159059
BathroomsTotalInteger                      0.071705
PropertyAge                                0.046907
LotSizeSquareFeet                          0.042159
district_Capistrano Unified                0.038280
district_Fremont Unified                   0.019791
district_San Ramon Valley Unified          0.017953
district_Irvine Unified                    0.015208
district_San Jose Unified                  0.012629
FireplaceYN                                0.011681
GarageSpaces                               0.010111
BedBathRatio                               0.009213
district_Poway Unified                     0.008066
district_Newport-Mesa Unified              0.006933
district_Santa Clara Unified               0.006832
district_San Bernardino City Unified       0.006637
district_San Diego Unified                 0.005702
district_Carlsbad Unified                  0.005644
district_Palos Verdes Peninsula Unified    0.005141
dtype: float64

Week 6 Model Comparison Notes
==============================
Test month: 202605
Train months: 202505, 202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604
Train size: 98567 rows

Results (test set):
  Linear Regression -> RMSE=$199,403  MAE=$135,081  R²=0.856
  Decision Tree      -> RMSE=$378,452  MAE=$271,583  R²=0.482
  Random Forest      -> RMSE=$364,152  MAE=$265,039  R²=0.520

Linear Regression
  Strengths: Fast to train, coefficients are directly interpretable
  (e.g. "$X per additional sqft" holding other features fixed), and it
  is a stable, low-variance baseline that's hard to overfit given
  enough rows relative to the number of zip-code/district dummy columns.
  Weaknesses: Assumes a linear, additive relationship between features
  and price. It can't capture interactions (e.g. an extra bedroom is
  worth more in some zip codes than others) or non-linear effects
  (e.g. diminishing returns on LivingArea past a certain size) unless
  those interactions are explicitly engineered as features.

Decision Tree
  Strengths: Captures non-linear relationships and feature interactions
  automatically (e.g. it can learn "zip=X AND LivingArea>2500 ->
  premium" without being told to). Also interpretable via its splits,
  and needs no feature scaling.
  Weaknesses: A single tree is high-variance and prone to overfitting,
  especially with many one-hot zip/district columns providing lots of
  ways to split on location alone. Small changes in the training data
  can produce a very different tree, and predictions are
  piecewise-constant (it can only predict values seen in training
  leaves), which tends to hurt RMSE/MAE versus a smoother model.

Random Forest
  Strengths: Averaging many decorrelated trees reduces the variance/
  overfitting problem of a single Decision Tree while keeping the
  ability to model non-linearities and interactions. Typically the best
  R² of the three here, and feature_importances_ gives a useful ranking
  of which fields (numeric features vs. specific zip/district dummies)
  drive price the most.
  Weaknesses: Slower to train and to run inference on than the other
  two models (300 trees vs. 1 tree vs. 1 linear model), less directly
  interpretable than a single tree or linear coefficients, and it still
  can't extrapolate outside the price/feature ranges seen in training
  (e.g. a brand-new luxury zip code with no sales history).

Feature engineering notes:
  - BedBathRatio (BathroomsTotalInteger / BedroomsTotal): rows with
    BedroomsTotal == 0 are dropped rather than assigned an infinite or
    arbitrary ratio.
  - PropertyAge (sale year - YearBuilt): computed using each month's
    sale year rather than today's date, so the feature reflects the
    property's age at time of transaction and stays consistent no
    matter when this script is re-run. Negative ages (data entry
    quirks) are clipped to 0.
  - DistrictName (via spatial join against Unified school district
    polygons): one-hot encoded alongside zip code. These two are
    collinear by construction (school districts and zip codes both
    encode location), so watch Linear Regression coefficient stability
    and compare feature_importances_ for zip_* vs district_* columns
    to see which one the tree models actually lean on.

Deliverable checklist:
  [x] Decision Tree and Random Forest regressors trained
  [x] Test R² compared against Linear Regression baseline
      (see model_comparison_results.csv)
  [x] Model behavior (strengths/weaknesses) documented above
  [x] School district spatial join (Unified districts only)
  [x] BedBathRatio and PropertyAge feature engineering

Saved to week5_model_notes.txt
  202606: removed 2320 outlier rows (12150 -> 9830)
Processed 202606: 9830 rows
  Warning: 73 zip code(s) in 202606 unseen during training (treated as no zip match): ['zip_95364', 'zip_94565-7901', 'zip_94523-3403', 'zip_94523-2326', 'zip_94703-1622', 'zip_95370-7928', 'zip_95222-9898', 'zip_94553-3575', 'zip_94061-1323', 'zip_95372-9705']...
  Warning: 1 district(s) in 202606 unseen during training (treated as no district match): ['district_Stony Creek Joint Unified']
  Linear Regression    RMSE=$194,138  MAE=$131,651  R²=0.858
  Decision Tree        RMSE=$376,875  MAE=$269,186  R²=0.466
  Random Forest        RMSE=$360,817  MAE=$261,438  R²=0.510

=== Evaluation on out-of-sample month: 202606 ===
            model          rmse           mae       r2
Linear Regression 194138.002219 131651.080204 0.858210
    Decision Tree 376875.430682 269186.277693 0.465656
    Random Forest 360816.948797 261438.405198 0.510222

# Table comparing old + new feature sets
![table](/Users/daisyzhang/IDX_ds/05_table.png)